In [1]:
%%bash
cat << 'EOF' > /content/config.sh
#!/bin/bash

export ROOTDIR="/content/exterieur"
export VIDEOSOURCE="gsplat/input/IMG_4797.MOV"
export IMAGESET="gsplat/input/perfume/video"
export INPUT_MODE="video"

#export NUM_FRAMES=150
export FPS=3

#branche dev
export GIT_BRANCH="dev"
export BASENAME="exterieur"

export PROFILE="gpu/quality"
#export PROFILE="gpu/balanced"
#export PROFILE="cpu/fast"
EOF

In [2]:
!cat /content/config.sh

#!/bin/bash

export ROOTDIR="/content/exterieur"
export VIDEOSOURCE="gsplat/input/IMG_4797.MOV"
export IMAGESET="gsplat/input/perfume/video"
export INPUT_MODE="video"

#export NUM_FRAMES=150
export FPS=3

#branche dev
export GIT_BRANCH="dev"
export BASENAME="exterieur"

export PROFILE="gpu/quality"
#export PROFILE="gpu/balanced"
#export PROFILE="cpu/fast"


In [10]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [11]:
!wget -q https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh
!bash Miniforge3-Linux-x86_64.sh -b -p /usr/local/miniforge

# activer conda pour cette session notebook
import os
os.environ["PATH"] = "/usr/local/miniforge/bin:" + os.environ["PATH"]

!conda --version

PREFIX=/usr/local/miniforge
Unpacking bootstrapper...
Unpacking payload...
Extracting ca-certificates-2026.2.25-hbd8a1cb_0.conda
Extracting libgomp-15.2.0-he0feb66_18.conda
Extracting libzlib-1.3.2-h25fd6f3_2.conda
Extracting nlohmann_json-abi-3.12.0-h0f90c79_1.conda
Extracting pybind11-abi-11-hc364b38_1.conda
Extracting python_abi-3.13-8_cp313.conda
Extracting tzdata-2025c-hc9c84f9_1.conda
Extracting _openmp_mutex-4.5-20_gnu.conda
Extracting zstd-1.5.7-hb78ec9c_6.conda
Extracting ld_impl_linux-64-2.45.1-default_hbd61a6d_101.conda
Extracting libgcc-15.2.0-he0feb66_18.conda
Extracting bzip2-1.0.8-hda65f42_9.conda
Extracting c-ares-1.34.6-hb03c661_0.conda
Extracting keyutils-1.6.3-hb9d3cd8_0.conda
Extracting libexpat-2.7.4-hecca717_0.conda
Extracting libffi-3.5.2-h3435931_0.conda
Extracting libgcc-ng-15.2.0-h69a702a_18.conda
Extracting libiconv-1.18-h3b78370_2.conda
Extracting liblzma-5.8.2-hb03c661_0.conda
Extracting libmpdec-4.0.0-hb03c661_1.conda
Extracting libstdcxx-15.2.0-h934c35e_1

In [13]:
!RUN=0; \
[ "$RUN" -eq 0 ] && echo "skipping this stage" || \
(wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh && \
chmod +x Miniconda3-latest-Linux-x86_64.sh  && \
bash Miniconda3-latest-Linux-x86_64.sh -b -p /usr/local/miniconda  && \
/usr/local/miniconda/bin/conda init bash)

skipping this stage


In [ ]:
%%bash
set -e
source /content/config.sh

mkdir -p "$ROOTDIR"

if [ -n "$VIDEOSOURCE" ]; then
  SRC_VIDEO="/content/drive/MyDrive/$VIDEOSOURCE"

  if [ ! -f "$SRC_VIDEO" ]; then
    echo "❌ Video not found: $SRC_VIDEO"
  else
    echo "🎬 Copying video: $SRC_VIDEO"
    cp -f "$SRC_VIDEO" "$ROOTDIR/video.mp4"
  fi
fi

if [ -n "$IMAGESET" ]; then
  SRC_IMAGES="/content/drive/MyDrive/$IMAGESET"
  DST_IMAGES="$ROOTDIR/images"

  if [ ! -d "$SRC_IMAGES" ]; then
    echo "❌ Image dataset not found: $SRC_IMAGES"
  else
    echo "🖼️ Preparing image dataset: $SRC_IMAGES"

    mkdir -p "$DST_IMAGES"

    COUNT=$(find "$SRC_IMAGES" -type f \( -iname "*.jpg" -o -iname "*.jpeg" -o -iname "*.png" \) | wc -l | tr -d ' ')
    if [ "$COUNT" -lt 2 ]; then
      echo "❌ Not enough images ($COUNT)"
      exit 1
    fi

    rm -f "$DST_IMAGES"/frame_*.png 2>/dev/null || true

    i=1
    for img in $(find "$SRC_IMAGES" -type f \( -iname "*.jpg" -o -iname "*.jpeg" -o -iname "*.png" \) | sort); do
      printf -v idx "%05d" "$i"
      cp "$img" "$DST_IMAGES/frame_${idx}.png"
      i=$((i+1))
    done

    echo "✅ Dataset ready in $DST_IMAGES"
  fi
fi

In [14]:
!git clone https://github.com/NicoIGN/video_to_ply.git
%cd video_to_ply

Cloning into 'video_to_ply'...
remote: Enumerating objects: 1127, done.
remote: Counting objects: 100% (134/134), done.
remote: Compressing objects: 100% (87/87), done.
remote: Total 1127 (delta 65), reused 103 (delta 38), pack-reused 993 (from 1)
Receiving objects: 100% (1127/1127), 689.85 KiB | 2.91 MiB/s, done.
Resolving deltas: 100% (715/715), done.
/content/video_to_ply


In [ ]:
%%bash
cd /content/video_to_ply
source /content/config.sh
git stash save && git checkout $GIT_BRANCH && git pull

In [ ]:
!RUN=0; \
[ "$RUN" -eq 0 ] && echo "skipping this stage" || \
( source /usr/local/miniforge/etc/profile.d/conda.sh && mamba env remove -y -n gsplat )

In [15]:
!source /usr/local/miniforge/etc/profile.d/conda.sh && \
mamba env list | grep -q "gsplat" && \
mamba env update -n gsplat -f environment/conda_colab.yml --prune -y || \
mamba env create -n gsplat -f environment/conda_colab.yml -y

[+] 0.0s
[+] 0.0s
[+] 0.1s
conda-forge/linux-64  ⣾  
conda-forge/noarch     1%[+] 0.2s
conda-forge/linux-64   5%
conda-forge/noarch    22%[+] 0.3s
conda-forge/linux-64  15%
conda-forge/noarch    41%[+] 0.4s
conda-forge/linux-64  21%
conda-forge/noarch    55%[+] 0.5s
conda-forge/linux-64  28%
conda-forge/noarch    69%[+] 0.6s
conda-forge/linux-64  35%
conda-forge/noarch    83%[+] 0.7s
conda-forge/linux-64  42%
conda-forge/noarch    90%conda-forge/noarch                                
[+] 0.8s
conda-forge/linux-64  53%[+] 0.9s
conda-forge/linux-64  58%[+] 1.0s
conda-forge/linux-64  65%[+] 1.1s
conda-forge/linux-64  73%[+] 1.2s
conda-forge/linux-64  78%[+] 1.3s
conda-forge/linux-64  85%[+] 1.4s
conda-forge/linux-64  89%[+] 1.5s
conda-forge/linux-64  97%conda-forge/linux-64                              


Transaction

  Prefix: /usr/local/miniforge/envs/gsplat

  Updating specs:

   - python=3.10
   - pip
   - rclone
   - tree
   - tqdm
   - cmake
   - ninja
   - numpy
   - scipy
   - eig

In [19]:
%%bash
source /usr/local/miniforge/etc/profile.d/conda.sh

SITE_PACKAGES=$(mamba run -n gsplat python -c "import site; print(site.getsitepackages()[0])")
TARGET="$SITE_PACKAGES/SuperGluePretrainedNetwork"

if [ ! -d "$TARGET" ]; then
    git clone --depth 1 \
        https://github.com/magicleap/SuperGluePretrainedNetwork.git \
        "$TARGET"
else
    echo "SuperGluePretrainedNetwork already installed."
fi

Cloning into '/usr/local/miniforge/envs/gsplat/lib/python3.10/site-packages/SuperGluePretrainedNetwork'...


In [20]:
!source /usr/local/miniforge/etc/profile.d/conda.sh && \
mamba run -n gsplat python -c "from SuperGluePretrainedNetwork.models import superpoint; print('SuperGluePretrainedNetwork OK')"

SuperGluePretrainedNetwork OK


In [21]:
!source /usr/local/miniforge/etc/profile.d/conda.sh && \
source /content/config.sh && unset NUM_FRAMES && \
echo INPUT_MODE=$INPUT_MODE && \
cd /content/video_to_ply/ && \
INPUT_ARG="" && \
if [ "$INPUT_MODE" = "images" ] && [ -n "$IMAGESET" ]; then \
  INPUT_ARG="--images $ROOTDIR/images --name $BASENAME"; \
elif [ "$INPUT_MODE" = "video" ] && [ -n "$VIDEOSOURCE" ]; then \
  INPUT_ARG="--video $ROOTDIR/video.mp4 --fps $FPS --name $BASENAME"; \
fi && \
mamba run -n gsplat bash run.sh $INPUT_ARG --root "$ROOTDIR" --skip-conda --profile "$PROFILE" --no-proxy

INPUT_MODE=video
🚫 Proxy disabled (NO_PROXY=true)
👉 using profile: gpu/quality
⏩ Skipping conda setup (--skip-conda enabled)
✅ Using python: Python 3.10.20 
🚀 GPU model OK: splatfacto
📦 ROOT: /content/exterieur
🎬 Extracting frames at 3 FPS → /content/exterieur/ori/images
🎬 Video:   /content/exterieur/input/video.mov
📏 Width:   1280px
📁 Output:  /content/exterieur/ori/images
⚙️ Mode:    Fixed FPS
🎞️ FPS:     3
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x57b2b9ab2400] moov atom not found
[in#0 @ 0x57b2b9ab2100] Error opening input: Invalid data found when processing input
Error opening input file /content/exterieur/input/video.mov.
Error opening input files: Invalid data found when processing input


In [ ]:
!RUN=0; \
[ "$RUN" -eq 0 ] && echo "skipping this stage" || \
( source /usr/local/miniforge/etc/profile.d/conda.sh && \
  source /content/config.sh && \
  cd /content/video_to_ply/ && \
  INPUT_ARG="" && \
  if [ "$INPUT_MODE" = "images" ] && [ -n "$IMAGESET" ]; then \
    INPUT_ARG="--images $ROOTDIR/images --name $BASENAME"; \
  elif [ "$INPUT_MODE" = "video" ] && [ -n "$VIDEOSOURCE" ]; then \
    INPUT_ARG="--video $ROOTDIR/video.mp4 --name $BASENAME"; \
  fi && \
  mamba run -n gsplat bash run.sh $INPUT_ARG \
    --root "$ROOTDIR" \
    --skip-conda \
    --profile "$PROFILE" \
    --no-proxy \
    --skip-conda \
    --skip-frame-extraction \
    --skip-colmap \
    --skip-training )

In [ ]:
%%bash
set -e
source /content/config.sh
cd $ROOTDIR/ori
zip -r $ROOTDIR/exports/colmap_$BASENAME.zip ./colmap

In [ ]:
from google.colab import files
import os
import subprocess

# =========================
# LOAD CONFIG.SH VARIABLES
# =========================
result = subprocess.run(
    "source /content/config.sh && env",
    shell=True,
    executable="/bin/bash",
    capture_output=True,
    text=True,
)

for line in result.stdout.splitlines():
    if "=" in line:
        key, value = line.split("=", 1)
        os.environ[key] = value

# =========================
# CONFIG
# =========================
rootdir = os.environ.get("ROOTDIR", "/content/work")
export_dir = os.path.join(rootdir, "exports")
basename = os.environ.get("BASENAME", "")

if not basename:
    print("❌ BASENAME is not set")
    raise SystemExit(1)

base_ply = os.path.join(export_dir, f"{basename}.ply")

# =========================
# EXPORT ORIGINAL PLY
# =========================
if os.path.exists(base_ply):
    print(f"⬇️ Downloading original PLY: {os.path.basename(base_ply)}")
    files.download(base_ply)
else:
    print(f"⚠️ Original PLY not found: {base_ply}")

In [ ]:
from google.colab import files
import os
import subprocess
import glob

# =========================
# LOAD CONFIG.SH VARIABLES
# =========================
result = subprocess.run(
    "source /content/config.sh && env",
    shell=True,
    executable="/bin/bash",
    capture_output=True,
    text=True,
)

for line in result.stdout.splitlines():
    if "=" in line:
        key, value = line.split("=", 1)
        os.environ[key] = value

# =========================
# CONFIG
# =========================
rootdir = os.environ.get("ROOTDIR", "/content/work")
export_dir = os.path.join(rootdir, "exports")
basename = os.environ.get("BASENAME", "")

if not basename:
    print("❌ BASENAME is not set")
    raise SystemExit(1)

# =========================
# FIND FILTERED PLYS
# =========================
filtered_plys = sorted(
    glob.glob(os.path.join(export_dir, f"{basename}_*.ply"))
)

# =========================
# EXPORT FILTERED PLYS
# =========================
if not filtered_plys:
    print("⚠️ No filtered PLY files found.")
    print(f"📂 Searched: {export_dir}")
else:
    print(f"📦 Found {len(filtered_plys)} filtered PLY file(s)")

    for ply_path in filtered_plys:
        print(f"⬇️ Downloading filtered PLY: {os.path.basename(ply_path)}")
        files.download(ply_path)